In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
from rdkit.DataStructs import TanimotoSimilarity

In [2]:
# Read the CSV file
df = pd.read_csv('sec_smiles.csv')

In [3]:
# Convert SMILES to RDKit molecule objects
df['Molecule'] = df['SMILES'].apply(lambda x: Chem.MolFromSmiles(x))

# Remove invalid molecules
df = df[df['Molecule'].notnull()].reset_index(drop=True)

[09:12:01] WARNING: not removing hydrogen atom without neighbors
[09:12:01] WARNING: not removing hydrogen atom without neighbors
[09:12:01] WARNING: not removing hydrogen atom without neighbors
[09:12:01] WARNING: not removing hydrogen atom without neighbors
[09:12:01] WARNING: not removing hydrogen atom without neighbors
[09:12:01] WARNING: not removing hydrogen atom without neighbors


In [4]:
# Compute Morgan fingerprints
radius = 2
nBits = 2048
morgan_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits) if mol is not None else None for mol in df['Molecule']]

[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerator
[09:12:05] DEPRECATION WARNING: please use MorganGenerat

In [5]:
num_molecules = len(morgan_fps)
similarity_matrix = np.zeros((num_molecules, num_molecules))
for i in range(num_molecules):
    for j in range(i, num_molecules):
        if morgan_fps[i] is not None and morgan_fps[j] is not None:
            sim = TanimotoSimilarity(morgan_fps[i], morgan_fps[j])
            similarity_matrix[i, j] = sim
            similarity_matrix[j, i] = sim  # Symmetric matrix
        else:
            similarity_matrix[i, j] = 0
            similarity_matrix[j, i] = 0

In [6]:
# Apply similarity threshold
threshold = 0.6
similarity_matrix[similarity_matrix < threshold] = 0

In [7]:
# Create a DataFrame for the similarity matrix
similarity_df = pd.DataFrame(similarity_matrix, index=df['ID'], columns=df['ID'])


In [8]:
# Display the similarity matrix
print(similarity_df)


ID            BS_0002   BS_0039   BS_0040  BS_0041  BS_0049  BS_0056  BS_0059  \
ID                                                                              
BS_0002           1.0  0.000000  0.000000      0.0      0.0      0.0      0.0   
BS_0039           0.0  1.000000  0.721519      0.0      0.0      0.0      0.0   
BS_0040           0.0  0.721519  1.000000      0.0      0.0      0.0      0.0   
BS_0041           0.0  0.000000  0.000000      1.0      0.0      0.0      0.0   
BS_0049           0.0  0.000000  0.000000      0.0      1.0      0.0      0.0   
...               ...       ...       ...      ...      ...      ...      ...   
BGC0002050_4      0.0  0.000000  0.000000      0.0      0.0      0.0      0.0   
BGC0001123_1      0.0  0.000000  0.000000      0.0      0.0      0.0      0.0   
BGC0000406_1      0.0  0.000000  0.000000      0.0      0.0      0.0      0.0   
BGC0000371_2      0.0  0.000000  0.000000      0.0      0.0      0.0      0.0   
BGC0002578_6      0.0  0.000

In [9]:
# Save the similarity matrix to CSV
similarity_df.to_csv('sec_similarity_matrix_morgan_all.csv')